# Boosting I - AdaBoost & Gradient Boosting
### Which streaming-service customers will churn?

Learn how boosting builds a strong predictor by training weak learners sequentially: each learner focuses on the examples or errors left by earlier learners.

> **Data required:** place `streaming_customers.csv` beside this notebook. The CSV is described as 8,000 streaming-service subscribers. It was not included with the supplied outline, so this notebook does not fabricate data.

We will implement AdaBoost with a weighted decision stump and gradient boosting for regression from scratch, compare both with scikit-learn models and Random Forest baselines, and tune a small Gradient Boosting classifier grid.

## Learning goals and dataset

The supplied schema describes these numeric input features:

- `tenure_months`, `monthly_bill_rs`, `support_calls_3m`, `late_payments_6m`
- `devices_linked`, `genres_watched`, `discount_active`, `app_crashes_last_month`

Targets: `churned` (binary classification, 0/1) and `monthly_watch_hours` (regression). We use one train/test split for every comparison. Imputation is fitted only on training data to avoid leaking test-set statistics.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import (
    AdaBoostClassifier,
    GradientBoostingClassifier,
    GradientBoostingRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split

RANDOM_STATE = 42
DATA_PATH = Path("streaming_customers.csv")
FEATURES = [
    "tenure_months",
    "monthly_bill_rs",
    "support_calls_3m",
    "late_payments_6m",
    "devices_linked",
    "genres_watched",
    "discount_active",
    "app_crashes_last_month",
]
CLASSIFICATION_TARGET = "churned"
REGRESSION_TARGET = "monthly_watch_hours"

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH.resolve()}. Place streaming_customers.csv beside this notebook."
    )

df = pd.read_csv(DATA_PATH)
required_columns = FEATURES + [CLASSIFICATION_TARGET, REGRESSION_TARGET]
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"CSV is missing required columns: {missing_columns}")

# The dataset specification describes numeric features and numeric targets.
df = df[required_columns].copy()
for column in required_columns:
    df[column] = pd.to_numeric(df[column], errors="raise")
if df[[CLASSIFICATION_TARGET, REGRESSION_TARGET]].isna().any().any():
    raise ValueError("Target columns cannot contain missing values.")
if not set(df[CLASSIFICATION_TARGET].unique()).issubset({0, 1}):
    raise ValueError("The churned target must contain only 0 and 1.")

print(f"Rows: {len(df):,}")
display(df.head())
display(df.describe().T)
print("Missing values per column:")
display(df.isna().sum())

## Train/test split and missing feature values

We stratify the classification split so the churn rate is represented in both partitions. The same row indices are then used for regression, making comparisons consistent. Median imputation is learned from each task's training features only.

In [ ]:
row_indices = np.arange(len(df))
train_indices, test_indices = train_test_split(
    row_indices,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df[CLASSIFICATION_TARGET],
)

X_train_raw = df.iloc[train_indices][FEATURES]
X_test_raw = df.iloc[test_indices][FEATURES]
X_train = SimpleImputer(strategy="median").fit_transform(X_train_raw)
X_test = SimpleImputer(strategy="median").fit(X_train_raw).transform(X_test_raw)

y_class_train = df.iloc[train_indices][CLASSIFICATION_TARGET].to_numpy(dtype=int)
y_class_test = df.iloc[test_indices][CLASSIFICATION_TARGET].to_numpy(dtype=int)
y_reg_train = df.iloc[train_indices][REGRESSION_TARGET].to_numpy(dtype=float)
y_reg_test = df.iloc[test_indices][REGRESSION_TARGET].to_numpy(dtype=float)

print(f"Training rows: {len(train_indices):,}; test rows: {len(test_indices):,}")
print(f"Churn rate - train: {y_class_train.mean():.3f}; test: {y_class_test.mean():.3f}")

## 1. Boosting versus bagging

**Bagging** trains many models independently on bootstrap samples, then averages/votes. Random Forest is a bagging method with additional feature randomness. **Boosting** trains models in sequence: later models focus on mistakes made by earlier models. This can reduce bias, but an overly complex or long sequence may overfit.

### AdaBoost intuition

AdaBoost increases the training weights of examples that are misclassified. Each weak learner's vote is weighted by its error: a learner with lower error gets more influence. Here the weak learner is a decision stump (a one-split tree).

In [ ]:
class WeightedDecisionStump:
    """A binary decision stump fitted by minimizing weighted Gini impurity."""

    def fit(self, X, y, sample_weight):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=int)
        sample_weight = np.asarray(sample_weight, dtype=float)
        self.majority_class_ = int(np.argmax(np.bincount(y, weights=sample_weight, minlength=2)))
        best = None
        total_weight = sample_weight.sum()

        for feature_index in range(X.shape[1]):
            order = np.argsort(X[:, feature_index], kind="mergesort")
            values = X[order, feature_index]
            labels = y[order]
            weights = sample_weight[order]
            split_positions = np.flatnonzero(values[:-1] < values[1:]) + 1
            if split_positions.size == 0:
                continue

            cumulative_class_weights = np.column_stack([
                np.cumsum(weights * (labels == class_label)) for class_label in (0, 1)
            ])
            left_weights = cumulative_class_weights[split_positions - 1]
            right_weights = cumulative_class_weights[-1] - left_weights
            left_totals = left_weights.sum(axis=1)
            right_totals = right_weights.sum(axis=1)

            def gini(weight_rows, totals):
                proportions = np.divide(
                    weight_rows,
                    totals[:, None],
                    out=np.zeros_like(weight_rows),
                    where=totals[:, None] > 0,
                )
                return totals * (1.0 - np.square(proportions).sum(axis=1))

            impurity = (gini(left_weights, left_totals) + gini(right_weights, right_totals)) / total_weight
            best_index = int(np.argmin(impurity))
            candidate = (float(impurity[best_index]), feature_index, int(split_positions[best_index]))
            if best is None or candidate < best:
                best = candidate

        if best is None:
            self.feature_index_ = None
            self.threshold_ = None
            self.left_class_ = self.right_class_ = self.majority_class_
        else:
            _, self.feature_index_, split_position = best
            sorted_values = np.sort(X[:, self.feature_index_], kind="mergesort")
            self.threshold_ = sorted_values[split_position - 1] + (
                sorted_values[split_position] - sorted_values[split_position - 1]
            ) / 2
            left = X[:, self.feature_index_] <= self.threshold_
            self.left_class_ = int(np.argmax(np.bincount(y[left], weights=sample_weight[left], minlength=2)))
            self.right_class_ = int(np.argmax(np.bincount(y[~left], weights=sample_weight[~left], minlength=2)))
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        if self.feature_index_ is None:
            return np.full(X.shape[0], self.majority_class_, dtype=int)
        return np.where(
            X[:, self.feature_index_] <= self.threshold_, self.left_class_, self.right_class_
        ).astype(int)


class ScratchAdaBoostClassifier:
    """Binary AdaBoost using weighted decision stumps (SAMME for two classes)."""

    def __init__(self, n_estimators=50):
        self.n_estimators = n_estimators

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=int)
        if set(np.unique(y)) != {0, 1}:
            raise ValueError("ScratchAdaBoostClassifier requires both classes encoded as 0 and 1.")
        sample_weight = np.full(len(y), 1.0 / len(y))
        self.estimators_ = []
        self.estimator_weights_ = []
        signed_y = 2 * y - 1
        epsilon = np.finfo(float).eps

        for _ in range(self.n_estimators):
            stump = WeightedDecisionStump().fit(X, y, sample_weight)
            predictions = stump.predict(X)
            error = float(sample_weight[predictions != y].sum())
            if error >= 0.5:
                break
            error = max(error, epsilon)
            alpha = 0.5 * np.log((1.0 - error) / error)
            self.estimators_.append(stump)
            self.estimator_weights_.append(alpha)
            if error <= epsilon:
                break
            sample_weight *= np.exp(-alpha * signed_y * (2 * predictions - 1))
            sample_weight /= sample_weight.sum()

        if not self.estimators_:
            raise ValueError("No stump achieved weighted error below 0.5; check the training data.")
        return self

    def predict(self, X):
        votes = np.zeros(len(X), dtype=float)
        for stump, weight in zip(self.estimators_, self.estimator_weights_):
            votes += weight * (2 * stump.predict(X) - 1)
        return (votes >= 0).astype(int)

scratch_ada = ScratchAdaBoostClassifier(n_estimators=50).fit(X_train, y_class_train)
scratch_ada_predictions = scratch_ada.predict(X_test)
print(f"Scratch AdaBoost stumps trained: {len(scratch_ada.estimators_)}")
print(classification_report(y_class_test, scratch_ada_predictions, zero_division=0))

### AdaBoost with scikit-learn

The library implementation provides the same broad idea with a well-tested estimator API. We keep the default decision stump as the weak learner. The custom implementation above makes the sample reweighting and weighted stump fit visible.

In [ ]:
sklearn_ada = AdaBoostClassifier(
    n_estimators=50,
    learning_rate=1.0,
    random_state=RANDOM_STATE,
)
sklearn_ada.fit(X_train, y_class_train)
sklearn_ada_predictions = sklearn_ada.predict(X_test)
print(classification_report(y_class_test, sklearn_ada_predictions, zero_division=0))

## 2. Gradient Boosting for regression, from scratch

For squared-error regression, the negative gradient of the loss is the residual (`actual - prediction`). Start with the training-target mean; then repeatedly fit a shallow regression tree to the residuals and add a fraction of its predictions. The `learning_rate` controls each tree's contribution.

In [ ]:
from sklearn.tree import DecisionTreeRegressor


class ScratchGradientBoostingRegressor:
    """Gradient boosting for squared-error regression using shallow regression trees."""

    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=2):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        self.initial_prediction_ = float(y.mean())
        current_prediction = np.full(len(y), self.initial_prediction_)
        self.estimators_ = []

        for _ in range(self.n_estimators):
            residuals = y - current_prediction
            tree = DecisionTreeRegressor(max_depth=self.max_depth, random_state=RANDOM_STATE)
            tree.fit(X, residuals)
            current_prediction += self.learning_rate * tree.predict(X)
            self.estimators_.append(tree)
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        prediction = np.full(X.shape[0], self.initial_prediction_)
        for tree in self.estimators_:
            prediction += self.learning_rate * tree.predict(X)
        return prediction

scratch_gbr = ScratchGradientBoostingRegressor(
    n_estimators=100, learning_rate=0.1, max_depth=2
).fit(X_train, y_reg_train)
scratch_gbr_predictions = scratch_gbr.predict(X_test)
print(f"Scratch Gradient Boosting test RMSE: {mean_squared_error(y_reg_test, scratch_gbr_predictions) ** 0.5:.3f}")
print(f"Scratch Gradient Boosting test R2:   {r2_score(y_reg_test, scratch_gbr_predictions):.3f}")

### Gradient Boosting with scikit-learn

Now compare the from-scratch regressor with the library implementation and a Random Forest regression baseline. We also evaluate churn classification using AdaBoost and a Random Forest.

In [ ]:
sklearn_gbr = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=2,
    random_state=RANDOM_STATE,
).fit(X_train, y_reg_train)
sklearn_gbr_predictions = sklearn_gbr.predict(X_test)

random_forest_classifier = RandomForestClassifier(
    n_estimators=200,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1,
).fit(X_train, y_class_train)
rf_class_predictions = random_forest_classifier.predict(X_test)

random_forest_regressor = RandomForestRegressor(
    n_estimators=200,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1,
).fit(X_train, y_reg_train)
rf_reg_predictions = random_forest_regressor.predict(X_test)

classification_models = {
    "Scratch AdaBoost": scratch_ada_predictions,
    "sklearn AdaBoost": sklearn_ada_predictions,
    "Random Forest": rf_class_predictions,
}
classification_results = pd.DataFrame([
    {
        "Model": name,
        "Accuracy": accuracy_score(y_class_test, predictions),
        "F1 (churn=1)": f1_score(y_class_test, predictions, zero_division=0),
    }
    for name, predictions in classification_models.items()
]).sort_values("F1 (churn=1)", ascending=False)

def regression_row(name, predictions):
    return {
        "Model": name,
        "RMSE": mean_squared_error(y_reg_test, predictions) ** 0.5,
        "MAE": mean_absolute_error(y_reg_test, predictions),
        "R2": r2_score(y_reg_test, predictions),
    }

regression_results = pd.DataFrame([
    regression_row("Scratch Gradient Boosting", scratch_gbr_predictions),
    regression_row("sklearn Gradient Boosting", sklearn_gbr_predictions),
    regression_row("Random Forest", rf_reg_predictions),
]).sort_values("RMSE")

display(classification_results.round(3))
display(regression_results.round(3))

## 3. Hyperparameter tuning

`n_estimators` controls the number of sequential trees, `learning_rate` shrinks each update, and `max_depth` controls each tree's complexity. We use cross-validation on the training partition only, then evaluate the selected model once on the held-out test set. The small grid is intended for demonstration, not exhaustive optimization.

In [ ]:
parameter_grid = {
    "n_estimators": [50, 100],
    "learning_rate": [0.05, 0.1],
    "max_depth": [1, 2],
}
tuning = GridSearchCV(
    GradientBoostingClassifier(random_state=RANDOM_STATE),
    param_grid=parameter_grid,
    scoring="f1",
    cv=3,
    n_jobs=-1,
    refit=True,
)
tuning.fit(X_train, y_class_train)
tuned_predictions = tuning.best_estimator_.predict(X_test)
print("Best parameters:", tuning.best_params_)
print(f"Cross-validation F1: {tuning.best_score_:.3f}")
print(f"Held-out test F1:    {f1_score(y_class_test, tuned_predictions, zero_division=0):.3f}")

## 4. Inspecting errors and feature importance

Metrics summarize performance, but confusion matrices show which churn outcomes are being missed. Tree-based feature importance is a useful first look, not proof of causality. Importance can be biased toward features with many possible split points.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

cm = confusion_matrix(y_class_test, tuned_predictions, labels=[0, 1])
axes[0].imshow(cm, cmap="Blues")
axes[0].set_title("Tuned Gradient Boosting: confusion matrix")
axes[0].set_xlabel("Predicted label")
axes[0].set_ylabel("True label")
axes[0].set_xticks([0, 1], labels=["Stay", "Churn"])
axes[0].set_yticks([0, 1], labels=["Stay", "Churn"])
for (row, column), value in np.ndenumerate(cm):
    axes[0].text(column, row, str(value), ha="center", va="center")

importances = pd.Series(tuning.best_estimator_.feature_importances_, index=FEATURES)
importances.sort_values().plot.barh(ax=axes[1], title="Feature importance")
axes[1].set_xlabel("Importance")
plt.tight_layout()
plt.show()

print(classification_report(y_class_test, tuned_predictions, target_names=["Stay", "Churn"], zero_division=0))

## 5. Bagging versus boosting: summary

| | Bagging (Random Forest) | Boosting (AdaBoost / Gradient Boosting) |
|---|---|---|
| Training | Many models fit independently | Models fit sequentially |
| Focus | Average away variance | Correct prior errors / follow residual gradients |
| Parallelism | Trees can be trained in parallel | The boosting sequence is inherently ordered |
| Main risk | Large forests can cost more memory/time | Deep trees, too many rounds, or high learning rates can overfit |

### Takeaways

- Compare models on held-out data; do not select a model by training score alone.
- Use F1/recall as well as accuracy when churn classes are imbalanced.
- In regression, RMSE penalizes large errors more than MAE; R2 is relative to a mean-prediction baseline.
- Tune tree depth, number of estimators, and learning rate using training-only cross-validation.
- The scratch examples illustrate the algorithms; production work should generally use maintained library implementations and validate them on representative data.